In [ ]:
import csv
import json
from datetime import datetime

In [ ]:
arquivo_csv = 'transacoes.csv'

In [ ]:
def ler_transacoes(arquivo):
  transacoes_validas = []
  try:
    with(open(arquivo, 'r', encoding="utf-8-sig")) as csv_file:
        csv_reader = csv.DictReader(csv_file)
        for transacao in csv_reader:
            registro = validar_transacao(transacao)
            if registro is not None:
              transacoes_validas.append(registro)

        RETORNO_TRANSACOES_VALIDAS = {
            "lista_transacoes": transacoes_validas,
            "total_linhas": csv_reader.line_num - 1,
            "linhas_validas": len(transacoes_validas),
            "linhas_invalidas": csv_reader.line_num - 1 - len(transacoes_validas)
        }
        print(f"Total de linhas lidas: {RETORNO_TRANSACOES_VALIDAS['total_linhas']}")
        print(f"Linhas válidas: {RETORNO_TRANSACOES_VALIDAS['linhas_validas']}")
        print(f"Linhas inválidas: {RETORNO_TRANSACOES_VALIDAS['linhas_invalidas']}")

        return RETORNO_TRANSACOES_VALIDAS

  except FileNotFoundError as e:
    print(f"Erro Encontrado: {e}")

In [ ]:
def valida_id(id):
  try:
    if int(id) > 0:
      return int(id)
    else:
      return False
  except Exception as e:
    return False

def valida_cliente_id(cliente_id):
  if cliente_id == None or cliente_id == '':
    return False
  else:
    return cliente_id

def valida_data(data):
    try:
        return datetime.strptime(data, '%Y-%m-%d').date()
    except (ValueError, TypeError):
        return False

def valida_tipo(tipo):
  if tipo in ('debito', 'credito'):
    return tipo
  else:
    return False

def valida_valor(valor):
  try:
    if float(valor) > 0:
      return float(valor)
    else:
      return False
  except Exception as e:
    return False

VALIDA_REGRAS = {
  'id': valida_id,
  'cliente_id': valida_cliente_id,
  'data': valida_data,
  'tipo': valida_tipo,
  'valor': valida_valor
}

def validar_transacao(transacao):
  registro_limpo = dict(transacao)

  for campo, funcao_validacao in VALIDA_REGRAS.items():

      resultado = funcao_validacao(transacao.get(campo))

      if resultado is False:
          return None
      registro_limpo[campo] = resultado

  return registro_limpo

In [ ]:
transacoes_validas = ler_transacoes(arquivo_csv)

In [ ]:
def gerar_relatorio(transacoes):

  LIMITE_SUSPEITO = 10000
  relatorios = []
  transacoes_suspeitas = []
  lista_transacoes = transacoes.get('lista_transacoes')

  datas = [t.get('data') for t in lista_transacoes]
  data_mais_antiga = min(datas)
  data_mais_recente = max(datas)
  dias_periodo = (data_mais_recente - data_mais_antiga).days

  for transacao in lista_transacoes:
    ano_mes = transacao.get('data').strftime('%Y-%m')

    if transacao.get('valor') > LIMITE_SUSPEITO:
      transacoes_suspeitas.append({
        'id': transacao.get('id'),
        'cliente_id': transacao.get('cliente_id'),
        'data': transacao.get('data').isoformat(),
        'valor': transacao.get('valor')
      })

    relatorio_existente = None
    for relatorio in relatorios:
      if ano_mes == relatorio.get('Mês'):
        relatorio_existente = relatorio
        break

    valor = transacao.get('valor')
    valor_credito = valor if transacao.get('tipo') == 'credito' else 0
    valor_debito = valor if transacao.get('tipo') == 'debito' else 0

    if relatorio_existente is None:
      relatorio_existente = {
        "Mês": ano_mes,
        "Transações": 1,
        "Total Crédito": valor_credito,
        "Total Débito": valor_debito,
        "Saldo": valor_credito - valor_debito,
        "Media": valor,
        "Maior Valor": valor,
        "Menor Valor": valor
      }
      relatorios.append(relatorio_existente)
    else:
      relatorio_existente['Transações'] += 1
      relatorio_existente['Total Crédito'] += valor_credito
      relatorio_existente['Total Débito'] += valor_debito
      relatorio_existente['Saldo'] += valor_credito - valor_debito
      relatorio_existente['Maior Valor'] = max(relatorio_existente['Maior Valor'], valor)
      relatorio_existente['Menor Valor'] = min(relatorio_existente['Menor Valor'], valor)

  for relatorio in relatorios:
    total_movimentado = relatorio['Total Crédito'] + relatorio['Total Débito']
    relatorio['Media'] = total_movimentado / relatorio['Transações']

  resumo_mensal = {}
  for relatorio in relatorios:
    ano_mes = relatorio.pop('Mês')
    resumo_mensal[ano_mes] = {
      'quantidade': relatorio['Transações'],
      'total_credito': relatorio['Total Crédito'],
      'total_debito': relatorio['Total Débito'],
      'saldo': relatorio['Saldo'],
      'media': relatorio['Media'],
      'maior_valor': relatorio['Maior Valor'],
      'menor_valor': relatorio['Menor Valor'],
    }

  relatorios_final = {
      "gerado_em": datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
      "total_transacoes_validas": transacoes.get('linhas_validas'),
      "total_transacoes_invalidas": transacoes.get('linhas_invalidas'),
      "periodo_analisado": {
          "data_mais_antiga": data_mais_antiga.isoformat(),
          "data_mais_recente": data_mais_recente.isoformat(),
          "dias_entre": dias_periodo
      },
      "resumo_mensal": resumo_mensal,
      "transacoes_suspeitas": transacoes_suspeitas
  }

  return relatorios_final

In [ ]:
relatorio = gerar_relatorio(transacoes_validas)

In [ ]:
def formatar_moeda(valor):
    """Formata número para o padrão brasileiro: R$ 1.234,56"""
    return f"R$ {valor:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")


def exibir_relatorio(relatorio):
    print("=" * 40)
    print("RELATÓRIO FINANCEIRO")
    print("=" * 40)

    print(f"Gerado em: {relatorio['gerado_em']}")
    print(f"Total de transações válidas:   {relatorio['total_transacoes_validas']}")
    print(f"Total de transações inválidas: {relatorio['total_transacoes_invalidas']}")

    periodo = relatorio['periodo_analisado']
    print(f"Período analisado: {periodo['data_mais_antiga']} → {periodo['data_mais_recente']} ({periodo['dias_entre']} dias)")

    print()
    print("=" * 40)
    print("RESUMO MENSAL")
    print("=" * 40)

    for ano_mes in sorted(relatorio['resumo_mensal'].keys()):
        dados = relatorio['resumo_mensal'][ano_mes]
        print(f"\nMês: {ano_mes}")
        print(f"  Transações:    {dados['quantidade']}")
        print(f"  Total crédito: {formatar_moeda(dados['total_credito'])}")
        print(f"  Total débito:  {formatar_moeda(dados['total_debito'])}")
        print(f"  Saldo:         {formatar_moeda(dados['saldo'])}")
        print(f"  Média:         {formatar_moeda(dados['media'])}")
        print(f"  Maior valor:   {formatar_moeda(dados['maior_valor'])}")
        print(f"  Menor valor:   {formatar_moeda(dados['menor_valor'])}")

    print()
    print("=" * 40)
    print("TRANSAÇÕES SUSPEITAS")
    print("=" * 40)

    suspeitas = relatorio['transacoes_suspeitas']
    if not suspeitas:
        print("Nenhuma transação suspeita encontrada.")
    else:
        for t in suspeitas:
            print(f"ID: {t['id']} | Cliente: {t['cliente_id']} | Data: {t['data']} | Valor: {formatar_moeda(t['valor'])}")

In [ ]:
def salvar_json(dados, caminho='relatorio.json'):
    with open(caminho, 'w', encoding='utf-8') as f:
        json.dump(dados, f, indent=2, ensure_ascii=False)
    print(f"Relatório salvo em: {caminho}")

In [ ]:
exibir_relatorio(relatorio)
salvar_json(relatorio)